# Sen1Floods11 — Flood Segmentation with U-Net
### Binary Segmentation: Flood vs Non-Flood from Sentinel-1 SAR

**Input:** S1 SAR (2 bands: VV, VH) — 512x512 chips  
**Output:** Binary flood mask (0 = non-flood, 1 = flood)  
**Model:** U-Net  
**Loss:** Dice + BCE (masked for invalid pixels)

---
**Sections:**
1. Imports & Config
2. Dataset & DataLoaders
3. U-Net Model
4. Loss Function
5. Training Loop
6. Evaluation (Test + Bolivia held-out)
7. Prediction Visualization

## 1. Imports & Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

from tqdm.auto import tqdm
import random
import copy

print(f"PyTorch version: {torch.__version__}")

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
DATA_ROOT = Path("/Users/hardikkamboj/Desktop/UMD/605/group_project/Sen1Floods11/v1.1")
S1_DIR    = DATA_ROOT / "data" / "flood_events" / "HandLabeled" / "S1Hand"
LABEL_DIR = DATA_ROOT / "data" / "flood_events" / "HandLabeled" / "LabelHand"
SPLIT_DIR = DATA_ROOT / "splits" / "flood_handlabeled"

# Normalization constants (from EDA — computed over 30 random chips)
VV_MEAN, VV_STD = -10.41, 4.14
VH_MEAN, VH_STD = -17.14, 4.68

# Training hyperparameters
BATCH_SIZE    = 4
EPOCHS        = 50
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
DICE_WEIGHT   = 0.5
BCE_WEIGHT    = 0.5
PATIENCE      = 10       # early stopping patience
LR_PATIENCE   = 5        # ReduceLROnPlateau patience

print("Config set ✓")
print(f"  Train CSV:  {SPLIT_DIR / 'flood_train_data.csv'}")
print(f"  S1 dir:     {S1_DIR}")
print(f"  Label dir:  {LABEL_DIR}")

## 2. Dataset & DataLoaders

In [ ]:
class Sen1Floods11Dataset(Dataset):
    """
    PyTorch Dataset for Sen1Floods11.
    Loads S1 SAR (VV, VH) as input and LabelHand as target.
    Handles NaN in S1, masks invalid pixels (-1) in labels.
    """

    def __init__(self, csv_path, s1_dir, label_dir, augment=False):
        self.s1_dir    = Path(s1_dir)
        self.label_dir = Path(label_dir)
        self.augment   = augment

        # Parse CSV: each line is "s1_file,label_file"
        self.pairs = []
        with open(csv_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                s1_name, label_name = line.split(",")
                self.pairs.append((s1_name.strip(), label_name.strip()))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        s1_name, label_name = self.pairs[idx]

        # ── Load S1 (2 bands: VV, VH) ──
        with rasterio.open(self.s1_dir / s1_name) as src:
            s1 = src.read().astype(np.float32)       # (2, 512, 512)

        # ── Load Label ──
        with rasterio.open(self.label_dir / label_name) as src:
            label = src.read(1).astype(np.float32)   # (512, 512)

        # ── Normalize S1 ──
        s1[0] = (s1[0] - VV_MEAN) / VV_STD
        s1[1] = (s1[1] - VH_MEAN) / VH_STD
        s1 = np.nan_to_num(s1, nan=0.0)             # NaN → 0 (= mean after norm)

        # ── Valid mask & clamp label ──
        valid_mask = (label != -1).astype(np.float32)
        label = np.clip(label, 0, 1)

        # ── Augmentation (training only) ──
        if self.augment:
            s1, label, valid_mask = self._augment(s1, label, valid_mask)

        # ── Convert to tensors ──
        image      = torch.from_numpy(s1.copy())                      # (2, H, W)
        label      = torch.from_numpy(label.copy()).unsqueeze(0)       # (1, H, W)
        valid_mask = torch.from_numpy(valid_mask.copy()).unsqueeze(0)  # (1, H, W)

        return {"image": image, "label": label, "valid_mask": valid_mask,
                "chip_id": s1_name.replace("_S1Hand.tif", "")}

    def _augment(self, image, label, valid_mask):
        """Random spatial augmentations applied identically to image + label + mask."""
        if random.random() > 0.5:
            image      = np.flip(image, axis=2)
            label      = np.flip(label, axis=1)
            valid_mask = np.flip(valid_mask, axis=1)
        if random.random() > 0.5:
            image      = np.flip(image, axis=1)
            label      = np.flip(label, axis=0)
            valid_mask = np.flip(valid_mask, axis=0)
        k = random.randint(0, 3)
        if k > 0:
            image      = np.rot90(image, k, axes=(1, 2))
            label      = np.rot90(label, k, axes=(0, 1))
            valid_mask = np.rot90(valid_mask, k, axes=(0, 1))
        return image, label, valid_mask

print("Dataset class defined ✓")

In [ ]:
# ── Create datasets ────────────────────────────────────────────────────────
train_ds   = Sen1Floods11Dataset(SPLIT_DIR / "flood_train_data.csv",   S1_DIR, LABEL_DIR, augment=True)
val_ds     = Sen1Floods11Dataset(SPLIT_DIR / "flood_valid_data.csv",   S1_DIR, LABEL_DIR, augment=False)
test_ds    = Sen1Floods11Dataset(SPLIT_DIR / "flood_test_data.csv",    S1_DIR, LABEL_DIR, augment=False)
bolivia_ds = Sen1Floods11Dataset(SPLIT_DIR / "flood_bolivia_data.csv", S1_DIR, LABEL_DIR, augment=False)

print(f"Train:    {len(train_ds)} chips")
print(f"Val:      {len(val_ds)} chips")
print(f"Test:     {len(test_ds)} chips")
print(f"Bolivia:  {len(bolivia_ds)} chips (held-out country)")

# ── Create DataLoaders ─────────────────────────────────────────────────────
train_loader   = DataLoader(train_ds,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader     = DataLoader(val_ds,     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader    = DataLoader(test_ds,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
bolivia_loader = DataLoader(bolivia_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Sanity check ───────────────────────────────────────────────────────────
batch = next(iter(train_loader))
print(f"\nSanity check (one train batch):")
print(f"  Image shape:      {batch['image'].shape}   dtype: {batch['image'].dtype}")
print(f"  Label shape:      {batch['label'].shape}   dtype: {batch['label'].dtype}")
print(f"  Valid mask shape:  {batch['valid_mask'].shape}")
print(f"  Image value range: [{batch['image'].min():.2f}, {batch['image'].max():.2f}]")
print(f"  Label unique:      {batch['label'].unique().tolist()}")
print(f"  Chip IDs:          {batch['chip_id']}")

## 3. U-Net Model

Standard U-Net with 4 encoder levels:
- **Encoder:** DoubleConv → MaxPool, channels: 2 → 64 → 128 → 256 → 512 → 1024
- **Decoder:** ConvTranspose2d (upsample) + skip connection + DoubleConv
- **Output:** 1 channel (logits, no sigmoid — applied in loss)

In [ ]:
class DoubleConv(nn.Module):
    """Conv2d → BN → ReLU → Conv2d → BN → ReLU"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    """MaxPool → DoubleConv"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x):
        return self.block(x)


class Up(nn.Module):
    """ConvTranspose2d + skip concat → DoubleConv"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1):
        super().__init__()
        self.inc   = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1   = Up(1024, 512)
        self.up2   = Up(512, 256)
        self.up3   = Up(256, 128)
        self.up4   = Up(128, 64)
        self.outc  = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x  = self.up1(x5, x4)
        x  = self.up2(x, x3)
        x  = self.up3(x, x2)
        x  = self.up4(x, x1)
        return self.outc(x)


model = UNet(in_channels=2, out_channels=1).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"U-Net created ✓")
print(f"  Total parameters:     {total_params:,}")
print(f"  Device:               {DEVICE}")

## 4. Loss Function

**Combined Dice + BCE loss** with invalid pixel masking:
- **BCE:** per-pixel loss, good for learning precise boundaries
- **Dice Loss:** overlap-based, naturally handles class imbalance (flood is only 9%)
- Both ignore pixels where `valid_mask == 0` (label was -1)

In [ ]:
class CombinedLoss(nn.Module):
    """Dice + BCE with masking for invalid pixels."""

    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight  = bce_weight
        self.smooth      = smooth

    def forward(self, logits, targets, valid_mask):
        # ── Masked BCE ──
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        bce_masked = (bce * valid_mask).sum() / (valid_mask.sum() + 1e-8)

        # ── Masked Dice ──
        probs  = torch.sigmoid(logits) * valid_mask
        target = targets * valid_mask
        intersection = (probs * target).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + target.sum() + self.smooth)

        return self.dice_weight * (1 - dice) + self.bce_weight * bce_masked


criterion = CombinedLoss(dice_weight=DICE_WEIGHT, bce_weight=BCE_WEIGHT)
print("Loss: Dice + BCE with masking ✓")

## 5. Training Loop

- **Optimizer:** Adam with weight decay
- **Scheduler:** ReduceLROnPlateau — halves LR when val loss stalls for 5 epochs
- **Early stopping:** stops if no improvement for 10 epochs
- **Best model:** saved in memory based on best validation IoU

In [ ]:
def compute_iou(logits, targets, valid_mask, threshold=0.5):
    """Compute IoU for the flood class on valid pixels only."""
    preds = (torch.sigmoid(logits) > threshold).float() * valid_mask
    targs = targets * valid_mask

    intersection = (preds * targs).sum()
    union = preds.sum() + targs.sum() - intersection
    return (intersection / (union + 1e-8)).item()


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_iou  = 0.0

    for batch in tqdm(loader, desc="  Train", leave=False):
        images     = batch["image"].to(device)
        labels     = batch["label"].to(device)
        valid_mask = batch["valid_mask"].to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels, valid_mask)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_iou  += compute_iou(logits, labels, valid_mask)

    n = len(loader)
    return running_loss / n, running_iou / n


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_iou  = 0.0

    for batch in tqdm(loader, desc="  Val", leave=False):
        images     = batch["image"].to(device)
        labels     = batch["label"].to(device)
        valid_mask = batch["valid_mask"].to(device)

        logits = model(images)
        loss   = criterion(logits, labels, valid_mask)

        running_loss += loss.item()
        running_iou  += compute_iou(logits, labels, valid_mask)

    n = len(loader)
    return running_loss / n, running_iou / n


print("Training functions defined ✓")

In [ ]:
# ── Optimizer & Scheduler ──────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE, verbose=True
)

# ── Training ───────────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": [], "lr": []}
best_val_iou  = 0.0
best_epoch    = 0
best_weights  = None
no_improve    = 0

print(f"Starting training for {EPOCHS} epochs...")
print(f"{'Epoch':>5} {'Train Loss':>11} {'Val Loss':>11} {'Train IoU':>11} {'Val IoU':>11} {'LR':>10}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss,   val_iou   = validate(model, val_loader, criterion, DEVICE)

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)
    history["lr"].append(current_lr)

    # Check improvement
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_epoch   = epoch
        best_weights = copy.deepcopy(model.state_dict())
        no_improve   = 0
        marker = " ★"
    else:
        no_improve += 1
        marker = ""

    print(f"{epoch:>5} {train_loss:>11.4f} {val_loss:>11.4f} {train_iou:>11.4f} {val_iou:>11.4f} {current_lr:>10.6f}{marker}")

    scheduler.step(val_iou)

    # Early stopping
    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

# Load best weights
model.load_state_dict(best_weights)
print(f"\nTraining complete. Best val IoU: {best_val_iou:.4f} at epoch {best_epoch}")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

# Loss
axes[0].plot(epochs_range, history["train_loss"], label="Train", color="#2196F3")
axes[0].plot(epochs_range, history["val_loss"],   label="Val",   color="#FF9800")
axes[0].axvline(best_epoch, color="red", linestyle="--", alpha=0.5, label=f"Best epoch ({best_epoch})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss vs Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(epochs_range, history["train_iou"], label="Train", color="#2196F3")
axes[1].plot(epochs_range, history["val_iou"],   label="Val",   color="#FF9800")
axes[1].axvline(best_epoch, color="red", linestyle="--", alpha=0.5, label=f"Best epoch ({best_epoch})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("IoU (flood class)")
axes[1].set_title("IoU vs Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation — Test Set + Bolivia Held-Out

In [ ]:
@torch.no_grad()
def evaluate_full(model, loader, device, threshold=0.5):
    """
    Full evaluation: aggregate + per-chip metrics.
    Returns overall metrics dict and list of per-chip IoUs.
    """
    model.eval()

    total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0
    chip_ious = []
    chip_ids  = []

    for batch in tqdm(loader, desc="  Eval", leave=False):
        images     = batch["image"].to(device)
        labels     = batch["label"].to(device)
        valid_mask = batch["valid_mask"].to(device)

        logits = model(images)
        preds  = (torch.sigmoid(logits) > threshold).float()

        # Per-chip metrics
        for i in range(images.size(0)):
            v = valid_mask[i].bool()
            p = preds[i, 0][v[0]]
            t = labels[i, 0][v[0]]

            tp = ((p == 1) & (t == 1)).sum().item()
            fp = ((p == 1) & (t == 0)).sum().item()
            fn = ((p == 0) & (t == 1)).sum().item()
            tn = ((p == 0) & (t == 0)).sum().item()

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_tn += tn

            chip_iou = tp / (tp + fp + fn + 1e-8)
            chip_ious.append(chip_iou)
            chip_ids.append(batch["chip_id"][i])

    # Aggregate metrics
    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall    = total_tp / (total_tp + total_fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = total_tp / (total_tp + total_fp + total_fn + 1e-8)
    accuracy  = (total_tp + total_tn) / (total_tp + total_tn + total_fp + total_fn + 1e-8)

    metrics = {
        "IoU": iou, "F1": f1, "Precision": precision,
        "Recall": recall, "Accuracy": accuracy,
    }
    return metrics, chip_ious, chip_ids


print("Evaluation function defined ✓")

In [ ]:
# ── Test set evaluation ────────────────────────────────────────────────────
test_metrics, test_chip_ious, test_chip_ids = evaluate_full(model, test_loader, DEVICE)

print("\n" + "=" * 50)
print("TEST SET RESULTS (90 chips, 10 countries)")
print("=" * 50)
for k, v in test_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

# ── Bolivia held-out evaluation ─────────────────────────────────────────────
bol_metrics, bol_chip_ious, bol_chip_ids = evaluate_full(model, bolivia_loader, DEVICE)

print("\n" + "=" * 50)
print("BOLIVIA HELD-OUT RESULTS (15 chips, unseen country)")
print("=" * 50)
for k, v in bol_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

# ── Side-by-side comparison ─────────────────────────────────────────────────
print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print(f"  {'Metric':<12} {'Test':>10} {'Bolivia':>10} {'Delta':>10}")
print("  " + "-" * 44)
for k in test_metrics:
    t, b = test_metrics[k], bol_metrics[k]
    print(f"  {k:<12} {t:>10.4f} {b:>10.4f} {b-t:>+10.4f}")

In [ ]:
# ── Per-chip IoU distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(test_chip_ious, bins=25, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].axvline(np.mean(test_chip_ious), color="red", linestyle="--",
                label=f"Mean: {np.mean(test_chip_ious):.3f}")
axes[0].set_xlabel("IoU per Chip")
axes[0].set_ylabel("Number of Chips")
axes[0].set_title("Test Set — Per-Chip IoU Distribution")
axes[0].legend()

axes[1].hist(bol_chip_ious, bins=10, color="#FF9800", edgecolor="white", alpha=0.85)
axes[1].axvline(np.mean(bol_chip_ious), color="red", linestyle="--",
                label=f"Mean: {np.mean(bol_chip_ious):.3f}")
axes[1].set_xlabel("IoU per Chip")
axes[1].set_ylabel("Number of Chips")
axes[1].set_title("Bolivia Held-Out — Per-Chip IoU Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Prediction Visualization

In [ ]:
def visualize_predictions(model, dataset, device, indices, threshold=0.5, title_prefix=""):
    """
    Visualize model predictions for selected chips.
    Columns: VV | VH | Ground Truth | Prediction | Difference
    """
    model.eval()
    n = len(indices)
    fig, axes = plt.subplots(n, 5, figsize=(22, 4.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["S1 — VV", "S1 — VH", "Ground Truth", "Prediction", "Difference"]
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]
        image      = sample["image"].unsqueeze(0).to(device)
        label      = sample["label"].numpy()[0]            # (H, W)
        valid_mask = sample["valid_mask"].numpy()[0]       # (H, W)
        chip_id    = sample["chip_id"]

        with torch.no_grad():
            logits = model(image)
            pred = (torch.sigmoid(logits) > threshold).cpu().numpy()[0, 0]  # (H, W)

        # De-normalize for display
        vv = sample["image"][0].numpy() * VV_STD + VV_MEAN
        vh = sample["image"][1].numpy() * VH_STD + VH_MEAN

        # Percentile stretch for display
        def stretch(arr):
            lo, hi = np.nanpercentile(arr, 2), np.nanpercentile(arr, 98)
            return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)

        # Color-code label and prediction
        def label_to_rgb(lbl, mask):
            rgb = np.zeros((*lbl.shape, 3))
            rgb[mask == 0]                 = [0.50, 0.50, 0.50]  # invalid
            rgb[(mask == 1) & (lbl == 0)]  = [0.95, 0.95, 0.95]  # non-flood
            rgb[(mask == 1) & (lbl == 1)]  = [0.12, 0.47, 0.71]  # flood
            return rgb

        # Difference map
        diff = np.zeros((*label.shape, 3))
        diff[valid_mask == 0]                                 = [0.50, 0.50, 0.50]  # invalid
        diff[(valid_mask == 1) & (pred == label)]             = [0.95, 0.95, 0.95]  # correct
        diff[(valid_mask == 1) & (pred == 1) & (label == 0)]  = [0.90, 0.30, 0.30]  # FP
        diff[(valid_mask == 1) & (pred == 0) & (label == 1)]  = [0.20, 0.70, 0.20]  # FN

        # Compute per-chip IoU
        v = valid_mask.astype(bool)
        tp = ((pred[v] == 1) & (label[v] == 1)).sum()
        fp = ((pred[v] == 1) & (label[v] == 0)).sum()
        fn = ((pred[v] == 0) & (label[v] == 1)).sum()
        iou = tp / (tp + fp + fn + 1e-8)

        axes[row, 0].imshow(stretch(vv), cmap="gray")
        axes[row, 0].set_ylabel(f"{chip_id}\nIoU: {iou:.3f}", fontsize=9, rotation=0,
                                labelpad=100, va="center")
        axes[row, 1].imshow(stretch(vh), cmap="gray")
        axes[row, 2].imshow(label_to_rgb(label, valid_mask))
        axes[row, 3].imshow(label_to_rgb(pred, valid_mask))
        axes[row, 4].imshow(diff)

    for ax in axes.flat:
        ax.axis("off")

    # Legends
    label_patches = [
        mpatches.Patch(color=[0.12, 0.47, 0.71], label="Flood"),
        mpatches.Patch(color=[0.95, 0.95, 0.95], label="Non-flood"),
        mpatches.Patch(color=[0.50, 0.50, 0.50], label="Invalid"),
    ]
    diff_patches = [
        mpatches.Patch(color=[0.95, 0.95, 0.95], label="Correct"),
        mpatches.Patch(color=[0.90, 0.30, 0.30], label="False Positive"),
        mpatches.Patch(color=[0.20, 0.70, 0.20], label="False Negative"),
    ]
    axes[-1, 2].legend(handles=label_patches, loc="lower right", fontsize=7)
    axes[-1, 4].legend(handles=diff_patches, loc="lower right", fontsize=7)

    fig.suptitle(f"{title_prefix}Model Predictions", fontsize=14, fontweight="bold", y=1.0)
    plt.tight_layout()
    plt.show()


print("Visualization function defined ✓")

In [ ]:
# ── Random test set predictions ────────────────────────────────────────────
random.seed(42)
test_indices = random.sample(range(len(test_ds)), 6)
visualize_predictions(model, test_ds, DEVICE, test_indices, title_prefix="Test Set — ")

In [ ]:
# ── Bolivia held-out predictions (all 15 chips) ────────────────────────────
bolivia_indices = list(range(len(bolivia_ds)))
visualize_predictions(model, bolivia_ds, DEVICE, bolivia_indices,
                      title_prefix="Bolivia (Unseen Country) — ")

In [ ]:
# ── Error analysis: best & worst chips ─────────────────────────────────────
# Sort test chips by IoU
sorted_test = sorted(zip(test_chip_ious, range(len(test_chip_ious)), test_chip_ids),
                     key=lambda x: x[0])

worst_3 = [idx for _, idx, _ in sorted_test[:3]]
best_3  = [idx for _, idx, _ in sorted_test[-3:]]

print("Worst 3 chips (lowest IoU):")
for iou, idx, cid in sorted_test[:3]:
    print(f"  {cid}: IoU = {iou:.4f}")

print("\nBest 3 chips (highest IoU):")
for iou, idx, cid in sorted_test[-3:]:
    print(f"  {cid}: IoU = {iou:.4f}")

print("\n── Worst performing chips ──")
visualize_predictions(model, test_ds, DEVICE, worst_3, title_prefix="Worst — ")

print("── Best performing chips ──")
visualize_predictions(model, test_ds, DEVICE, best_3, title_prefix="Best — ")

## 8. Save Model

In [ ]:
# ── Save best model weights ────────────────────────────────────────────────
save_path = Path("/Users/hardikkamboj/Desktop/UMD/605/group_project/unet_flood_best.pt")
torch.save({
    "model_state_dict": best_weights,
    "epoch": best_epoch,
    "val_iou": best_val_iou,
    "config": {
        "vv_mean": VV_MEAN, "vv_std": VV_STD,
        "vh_mean": VH_MEAN, "vh_std": VH_STD,
        "batch_size": BATCH_SIZE, "lr": LR,
    },
    "test_metrics": test_metrics,
    "bolivia_metrics": bol_metrics,
}, save_path)

print(f"Model saved to {save_path}")
print(f"  Best epoch:    {best_epoch}")
print(f"  Best val IoU:  {best_val_iou:.4f}")
print(f"  Test IoU:      {test_metrics['IoU']:.4f}")
print(f"  Bolivia IoU:   {bol_metrics['IoU']:.4f}")